# HW4 — Environment Check

UCU Artificial Intelligence 2026

## Objectives

- Verify the Kaggle environment and GPU availability.
- Verify access to the competition dataset.
- Verify Hugging Face authentication.
- Verify access to Gemma, MamayLM, and Qwen3.

In [1]:
import os
import sys
from pathlib import Path
from importlib.metadata import version, PackageNotFoundError

import numpy as np
import pandas as pd
import torch

print("=== ENVIRONMENT ===")
print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("PyTorch:", torch.__version__)

print("\n=== PACKAGE VERSIONS ===")

packages = [
    "transformers",
    "huggingface-hub",
    "accelerate",
    "bitsandbytes",
    "peft",
    "trl",
    "unsloth",
    "vllm"
]

for package in packages:
    try:
        print(f"{package}: {version(package)}")
    except PackageNotFoundError:
        print(f"{package}: Not installed")

=== ENVIRONMENT ===
Python: 3.13.16
NumPy: 2.1.3
Pandas: 2.3.3
PyTorch: 2.11.0+cu130

=== PACKAGE VERSIONS ===
transformers: 5.20.2
huggingface-hub: 1.33.0
accelerate: 1.15.0
bitsandbytes: Not installed
peft: 0.21.2
trl: Not installed
unsloth: Not installed
vllm: Not installed


In [2]:
print("=== GPU CHECK ===")

cuda_available = torch.cuda.is_available()

print("CUDA available:", cuda_available)
print("CUDA version:", torch.version.cuda)
print("GPU count:", torch.cuda.device_count())

if cuda_available:
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)

        print(f"\nGPU {i}: {props.name}")
        print(f"VRAM: {props.total_memory / 1024**3:.2f} GiB")
        print(f"Compute capability: {props.major}.{props.minor}")

    print("\nBF16 supported:", torch.cuda.is_bf16_supported())

else:
    print("GPU is not available.")

=== GPU CHECK ===
CUDA available: True
CUDA version: 13.0
GPU count: 2

GPU 0: Tesla T4
VRAM: 14.56 GiB
Compute capability: 7.5

GPU 1: Tesla T4
VRAM: 14.56 GiB
Compute capability: 7.5

BF16 supported: True


In [3]:
INPUT_DIR = Path("/kaggle/input")

required_files = [
    "train.csv",
    "test.csv",
    "sample_submission.csv"
]

data_paths = {}

print("=== DATASET FILES ===")

for filename in required_files:
    matches = list(INPUT_DIR.rglob(filename))

    if len(matches) == 1:
        data_paths[filename] = matches[0]
        print(f"{filename}: FOUND")
        print(f"Path: {matches[0]}\n")

    elif len(matches) == 0:
        raise FileNotFoundError(f"{filename} not found")

    else:
        raise ValueError(
            f"Multiple files named {filename}: {matches}"
        )

DATA_DIR = data_paths["train.csv"].parent

print("Data directory:", DATA_DIR)

=== DATASET FILES ===
train.csv: FOUND
Path: /kaggle/input/competitions/ukrainian-school-math-slm-hw4-ucu-ai-2026/train.csv

test.csv: FOUND
Path: /kaggle/input/competitions/ukrainian-school-math-slm-hw4-ucu-ai-2026/test.csv

sample_submission.csv: FOUND
Path: /kaggle/input/competitions/ukrainian-school-math-slm-hw4-ucu-ai-2026/sample_submission.csv

Data directory: /kaggle/input/competitions/ukrainian-school-math-slm-hw4-ucu-ai-2026


In [4]:
train = pd.read_csv(
    data_paths["train.csv"],
    dtype=str,
    keep_default_na=False
)

test = pd.read_csv(
    data_paths["test.csv"],
    dtype=str,
    keep_default_na=False
)

sample_submission = pd.read_csv(
    data_paths["sample_submission.csv"],
    dtype=str,
    keep_default_na=False
)

print("=== DATASET CHECK ===")

print("Train shape:", train.shape)
print("Test shape:", test.shape)
print("Sample submission shape:", sample_submission.shape)

assert train.shape == (1526, 8)
assert test.shape == (3561, 4)
assert sample_submission.shape == (3561, 2)

assert train["id"].is_unique
assert test["id"].is_unique
assert sample_submission["id"].is_unique

assert set(train["id"]).isdisjoint(test["id"])
assert set(test["id"]) == set(sample_submission["id"])

assert list(sample_submission.columns) == ["id", "answer"]

print("\nNumber of train skills:", train["skill"].nunique())
print("Number of test skills:", test["skill"].nunique())

print("\nAll dataset checks passed.")

=== DATASET CHECK ===
Train shape: (1526, 8)
Test shape: (3561, 4)
Sample submission shape: (3561, 2)

Number of train skills: 29
Number of test skills: 29

All dataset checks passed.


In [5]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login, HfApi

secrets = UserSecretsClient()

HF_TOKEN = secrets.get_secret("HF_TOKEN")

login(
    token=HF_TOKEN,
    add_to_git_credential=False
)

api = HfApi()

user_info = api.whoami(token=HF_TOKEN)

print("=== HUGGING FACE ===")
print("Username:", user_info["name"])
print("Authentication successful.")

=== HUGGING FACE ===
Username: vikhimiak
Authentication successful.


In [6]:
from huggingface_hub import hf_hub_download

MODELS = {
    "Gemma": "google/gemma-3-4b-it",

    "MamayLM": (
        "INSAIT-Institute/"
        "MamayLM-Gemma-3-4B-IT-v1.0"
    ),

    "Qwen3": "Qwen/Qwen3-4B"
}

access_results = []

for name, model_id in MODELS.items():
    try:
        config_path = hf_hub_download(
            repo_id=model_id,
            filename="config.json",
            token=HF_TOKEN
        )

        access_results.append({
            "model": name,
            "model_id": model_id,
            "access": "OK"
        })

    except Exception as e:
        access_results.append({
            "model": name,
            "model_id": model_id,
            "access": f"FAILED: {type(e).__name__}"
        })

access_df = pd.DataFrame(access_results)

display(access_df)

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

,model,model_id,access
0,Gemma,google/gemma-3-4b-it,OK
1,MamayLM,INSAIT-Institute/MamayLM-Gemma-3-4B-IT-v1.0,OK
2,Qwen3,Qwen/Qwen3-4B,OK


In [7]:
checks = {
    "CUDA available": torch.cuda.is_available(),

    "Dataset available": (
        len(train) == 1526
        and len(test) == 3561
        and len(sample_submission) == 3561
    ),

    "Hugging Face authenticated": bool(user_info.get("name")),

    "All model configs accessible": (
        access_df["access"] == "OK"
    ).all()
}

print("=== FINAL ENVIRONMENT VALIDATION ===\n")

for check, passed in checks.items():
    status = "PASS" if passed else "FAIL"
    print(f"[{status}] {check}")

assert all(checks.values()), "Environment validation failed."

print("\nEnvironment is ready.")

=== FINAL ENVIRONMENT VALIDATION ===

[PASS] CUDA available
[PASS] Dataset available
[PASS] Hugging Face authenticated
[PASS] All model configs accessible

Environment is ready.
